Logistic Regression using entire datasets


In [ ]:
#Import the necessary libraries
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import shuffle

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


In [ ]:
max_length = 35
max_tokens = 10000
text_vectorization = layers.TextVectorization(max_tokens=max_tokens,
                                              output_mode='int',
                                              output_sequence_length=max_length,
                                              standardize='strip_punctuation',
                                              split='whitespace')


In [ ]:
#Convert train data to tf.data.Dataset object
BATCH = 32
BUFFER_SIZE = len(X_train_mid) # Use X_train_mid for buffer size
train_tf = tf.data.Dataset.from_tensor_slices((
    X_train_mid, # Use X_train_mid
    y_train_mid # Use y_train_mid
))

train_tf = train_tf.shuffle(BUFFER_SIZE, seed=SEED, reshuffle_each_iteration=False).batch(BATCH)


#Convert test data to tf.data.Dataset object
test_tf = tf.data.Dataset.from_tensor_slices(X_test_mid) # Use X_test_mid

#Convert the data into batch
test_tf = test_tf.batch(BATCH)



In [ ]:
#Learn the vocabulary
text_vectorization.adapt(train_tf.map(lambda twt, target: twt))
# twt: keyword + location + text (문자열)
# target: 정답 레이블 (0 또는 1)
#즉, train_tf는 .map으로 각 튜플에서 텍스트 부분(twt)만 뽑음
#adapt는, 자주 등장하는 단어를 정수 인덱스로 매핑한 사전을 만든다.

In [ ]:
# Display the maximum sequence length being used
print(f"Maximum sequence length (max_length): {max_length}")

In [ ]:
#Define a Transformer Encoder using subclassed layer
#keras의 Layer를 서브 클래싱하여 직접 정의
from keras.saving import register_keras_serializable
@register_keras_serializable()
class TransformerEncoder(layers.Layer):
    supports_masking = True  # 마스크 지원 명시
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        #Size of the input vector (size of the vocabulary)
        self.embed_dim = embed_dim
        #Size of the inner dense layer
        self.dense_dim = dense_dim
        #Number of attention heads
        self.num_heads = num_heads

        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim//num_heads)
        self.dense_proj = keras.Sequential(
                   [layers.Dense(dense_dim, activation="relu"),
                    layers.Dense(embed_dim),]
                                        )
        #relu활성화, 다시 embed_dim 으로 차원 축소

        self.layernorm_1 = layers.LayerNormalization() #Residual + Norm 구조
        self.layernorm_2 = layers.LayerNormalization()
        self.max_pool1 = layers.GlobalMaxPooling1D()

    #Define a call() method  where forward pass is implemented
    def call(self, inputs, mask=None): #pytorch의 feedforward
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        #마스크 처리 - attention mask가 주어지면 shape을 맞춰준다.

        #Apply the attention layer
        attention_output = self.attention(inputs, inputs, attention_mask=mask)
        #Normalize the data
        proj_input = self.layernorm_1(inputs + attention_output)
        #Apply the dense layer
        proj_output = self.dense_proj(proj_input)
        #Normalize the data and return it
        return self.layernorm_2(proj_input + proj_output)

        #return self.max_pool1(norm)

    #Define configuration method
    # 모델 저장/불러오기용 설정값 반환
    def get_config(self):
        config = super().get_config()
        config.update({
                    "embed_dim": self.embed_dim,
                    "num_heads": self.num_heads,
                    "dense_dim": self.dense_dim,
                    })
        return config

In [ ]:
# Implementing positional embedding as a subclassed layer
from keras import ops
from keras.saving import register_keras_serializable

@register_keras_serializable()
class PositionalEmbedding(layers.Layer): #input_dim:정수 개수
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.input_dim = input_dim #어휘 사전 크기
        self.output_dim = output_dim #출력 임베딩 차원, 하나의 토큰이 해당 크기의 차원으로 매핑//////임베
        self.token_embeddings = layers.Embedding(
          input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
                   input_dim=sequence_length, output_dim=output_dim)

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        #정수 시퀀스를 실수 벡터 시퀀스로 변환 ← 이게 바로 Embedding
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions


    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)
    #inputs 안에 있는 0은 실제 단어가 아니라 패딩이기 때문에,
    #모델이 그 위치의 정보를 학습하거나 계산에 반영하지 않도록 마스킹 처리가 필요
    #0은 False로 처리해서, attention의 대상에서 제외시킴.

    def get_config(self):
        config = super().get_config()
        config.update({
           "output_dim": self.output_dim,
           "sequence_length": self.sequence_length,
           "input_dim": self.input_dim,
                    })
        return config


In [ ]:
# === 3. 콜백 정의 (학습률 스케줄러) ===
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',      # 검증 손실을 기준으로 학습률 감소
    factor=0.5,              # 감소 비율 (예: lr = lr * 0.5)
    patience=10,              # 몇 epoch 동안 개선 없으면 감소
    min_lr=1e-7,             # 최소 학습률
    verbose=1
)

In [ ]:
#Construct the model

#Define the input
inputs = keras.Input(shape=(None,), dtype="int64")

pos_embed = PositionalEmbedding(sequence_length=35,
                        input_dim=10000,#사전크기_구간별로 유동적
                        output_dim=256)(inputs) 

#Apply the encoder
encoded = TransformerEncoder(embed_dim=256,
                             dense_dim=32,
                             num_heads=8)(pos_embed)


x = layers.GlobalMaxPooling1D()(encoded)
x = layers.Dropout(0.5)(x)
output = layers.Dense(units=1, activation="sigmoid")(x)
#마지막에 이진 분류 형태로 모델링

model = keras.Model(inputs=inputs,outputs=output)


In [ ]:
#Define the validation data size
val_size = int(0.1 * len(train_tf))

In [ ]:
#Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, beta_1=0.9, beta_2=0.98,epsilon=1e-9),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy']
             )

In [ ]:
#Train the model

# Calculate class weights based on y_train_mid distribution
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_mid)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y_train_mid)
class_weight = dict(zip(classes, class_weights_array))

print("Calculated Class Weights:", class_weight)

history = model.fit(train_data_mid, # Changed from train_data to train_data_mid
                    epochs=100,
                    validation_data=validation_data_mid, # Changed from validation_data to validation_data_mid
                    #callbacks=callbacks, # Commented out as callbacks was commented out previously
                    class_weight=class_weight # Uncommented and using calculated class_weight
                    )
#경고문 무시가능. 어텐션에서는 마스킹 적용됨.

In [ ]:
# Vectorize the X_test data using the adapted text_vectorization layer
X_test_vectorized = text_vectorization(X_test_mid)

# Classify the tweets of X_test data using the transformer model
predictions = model.predict(X_test_vectorized)

# Print the predictions
print("Predictions for X_test:")
display(predictions)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# Ensure predictions are a 1D array if they are not already
# model.predict can sometimes return a shape like (n_samples, 1)
if predictions.shape[-1] == 1:
    predictions = predictions.flatten()

# Convert predictions to binary labels using a threshold (e.g., 0.5)
threshold = 0.5
binary_predictions = (predictions > threshold).astype(int)

# Calculate and print accuracy
accuracy = accuracy_score(y_test_mid, binary_predictions)
print(f"Accuracy on X_test: {accuracy:.4f}")

# Print classification report
print("\nClassification Report on X_test:")
print(classification_report(y_test_mid, binary_predictions, zero_division=0)) # Add zero_division to handle cases with no predicted samples

# Calculate and print precision, recall, and F1-score
precision = precision_score(y_test_mid, binary_predictions, zero_division=0)
recall = recall_score(y_test_mid, binary_predictions, zero_division=0)
f1 = f1_score(y_test_mid, binary_predictions, zero_division=0)

print(f"\nPrecision on X_test: {precision:.4f}")
print(f"Recall on X_test: {recall:.4f}")
print(f"F1-score on X_test: {f1:.4f}")


# Calculate and display confusion matrix
cm = confusion_matrix(y_test, binary_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix on X_test")
plt.show()